# Sampling basis

**Contributed by**: Benoît Legat

In [1]:
using DynamicPolynomials
using SumOfSquares
import MultivariateBases as MB

using Dualization
import Hypatia
import SCS

In this tutorial, we show how to use a sampling basis for enforcing
the equality between the polynomial and its Sum-of-Squares decomposition.
As introduced in Zero basis, we can set the zero basis to a sampling
basis using the `zero_basis` keyword.

In [2]:
@polyvar x
p = x^4 - 4x^3 - 2x^2 + 12x + 3

model = Model(dual_optimizer(SCS.Optimizer))
set_silent(model)
@variable(model, γ)
@objective(model, Max, γ)
@constraint(model, p - γ in SOSCone(), zero_basis = BoxSampling([-1], [1]))
optimize!(model)
solution_summary(model)

solution_summary(; result = 1, verbose = false)
├ solver_name          : Dual model with SCS attached
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : solved
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : -5.99959e+00
│ └ dual_objective_value : -6.00019e+00
└ Work counters
  └ solve_time (sec)   : 1.12935e-03

We can see that the SOS constraint is converted into a PSD constraint.

In [3]:
print_active_bridges(model)

 * Supported objective: MOI.VariableIndex
 * Unsupported constraint: MOI.VectorAffineFunction{Float64}-in-SumOfSquares.SOSPolynomialSet{FullSpace, StarAlgebras.SubBasis{MultivariateBases.Polynomial{Monomial, Vector{DynamicPolynomials.Variable{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}}}, Vector{Int64}}, Int64, Vector{Int64}, StarAlgebras.MappedBasis{MultivariateBases.Polynomial{Monomial, Vector{DynamicPolynomials.Variable{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}}}, Vector{Int64}}, Vector{Int64}, MultivariatePolynomials.ExponentsIterator{MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}, Nothing, Vector{Int64}}, MultivariateBases.Variables{Monomial, Vector{DynamicPolynomials.Variable{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, MultivariatePolynomials.Graded{MultivariatePolyno

Let's try with Hypatia now:

In [4]:
set_optimizer(model, dual_optimizer(Hypatia.Optimizer))
optimize!(model)
solution_summary(model)

solution_summary(; result = 1, verbose = false)
├ solver_name          : Dual model with Hypatia attached
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : Optimal
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : -6.00000e+00
│ └ dual_objective_value : -6.00000e+00
└ Work counters
  ├ solve_time (sec)   : 8.33911e+00
  └ barrier_iterations : 7

We can see that the SOS constraint is passed as a
`LowRankOpt.SetDotProducts{LowRankOpt.WITHOUT_SET}`. This maps into
Hypatia's native `WSOSInterpNonnegativeCone`.

In [5]:
print_active_bridges(model)

 * Supported objective: MOI.VariableIndex
 * Unsupported constraint: MOI.VectorAffineFunction{Float64}-in-SumOfSquares.SOSPolynomialSet{FullSpace, StarAlgebras.SubBasis{MultivariateBases.Polynomial{Monomial, Vector{DynamicPolynomials.Variable{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}}}, Vector{Int64}}, Int64, Vector{Int64}, StarAlgebras.MappedBasis{MultivariateBases.Polynomial{Monomial, Vector{DynamicPolynomials.Variable{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}}}, Vector{Int64}}, Vector{Int64}, MultivariatePolynomials.ExponentsIterator{MultivariatePolynomials.Graded{MultivariatePolynomials.LexOrder}, Nothing, Vector{Int64}}, MultivariateBases.Variables{Monomial, Vector{DynamicPolynomials.Variable{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, MultivariatePolynomials.Graded{MultivariatePolyno

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*